# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import duckdb
import os, sys

# Setup repository path safely for Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
con = duckdb.connect()

# 1. Bucket Table for Signal 1: Content Age (Staleness)
q_s1 = """
SELECT
    CASE
        WHEN content_age_days < 90 THEN '01: <90d'
        WHEN content_age_days BETWEEN 90 AND 180 THEN '02: 90-180d'
        WHEN content_age_days BETWEEN 181 AND 365 THEN '03: 181-365d'
        ELSE '04: >365d'
    END as age_bucket,
    COUNT(*) as n,
    ROUND(AVG(CASE WHEN trend_direction = 'down' THEN 1.0 ELSE 0.0 END) * 100, 2) as decline_rate_pct
FROM df
GROUP BY 1 ORDER BY 1;
"""
print("--- Signal 1: Staleness Bucket Table ---")
print(con.execute(q_s1).df())

# 2. Bucket Table for Signal 2: Impressions Bucket
q_s2 = """
SELECT
    CASE
        WHEN impressions_90d < 500 THEN '01: Low (<500)'
        WHEN impressions_90d BETWEEN 500 AND 2000 THEN '02: Medium (500-2k)'
        WHEN impressions_90d BETWEEN 2001 AND 10000 THEN '03: High (2k-10k)'
        ELSE '04: Very High (>10k)'
    END as impression_bucket,
    COUNT(*) as n,
    ROUND(AVG(CASE WHEN trend_direction = 'down' THEN 1.0 ELSE 0.0 END) * 100, 2) as decline_rate_pct
FROM df
GROUP BY 1 ORDER BY 1;
"""
print("\n--- Signal 2: Impressions Bucket Table ---")
print(con.execute(q_s2).df())

--- Signal 1: Staleness Bucket Table ---
     age_bucket      n  decline_rate_pct
0   02: 90-180d  12272             62.73
1  03: 181-365d  11368             51.49
2     04: >365d   6360             42.63

--- Signal 2: Impressions Bucket Table ---
      impression_bucket      n  decline_rate_pct
0        01: Low (<500)  13274             47.47
1   02: Medium (500-2k)   6513             61.77
2     03: High (2k-10k)   6611             61.29
3  04: Very High (>10k)   3602             52.36


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# Create outputs folder if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# 1. Compute Action Score, Reason Code, and Action Label
df_queue = df.copy()

# Score formula: log(1 + impressions) * (content_age_days / 365)
df_queue['baseline_action_score'] = np.log1p(df_queue['impressions_90d']) * (df_queue['content_age_days'] / 365.0)

# Apply Rule Logic
def assign_action_and_reason(row):
    if row['content_age_days'] >= 90 and row['impressions_90d'] >= 500:
        return 'REFRESH_CONTENT', 'STALE_HIGH_IMPRESSIONS'
    elif row['content_age_days'] >= 180:
        return 'REVIEW_STALE', 'STALE_LOW_IMPRESSIONS'
    else:
        return 'MONITOR', 'LOW_PRIORITY'

res = df_queue.apply(assign_action_and_reason, axis=1)
df_queue['action_label'] = [r[0] for r in res]
df_queue['reason_code'] = [r[1] for r in res]

# Sort by highest baseline score first
df_queue = df_queue.sort_values(by='baseline_action_score', ascending=False).reset_index(drop=True)

# Select required columns for output CSV
output_cols = ['content_id', 'client_id', 'baseline_action_score', 'reason_code', 'action_label', 'impressions_90d', 'content_age_days']
csv_df = df_queue[output_cols]

# 2. Write CSV file
output_path = 'work/outputs/baseline_action_score.csv'
csv_df.to_csv(output_path, index=False)

print(f"✅ Successfully written ranked queue to '{output_path}'")
print(f"Total rows in queue: {len(csv_df)}")
print("\nTop 5 rows preview:")
print(csv_df.head(5))

✅ Successfully written ranked queue to 'work/outputs/baseline_action_score.csv'
Total rows in queue: 30000

Top 5 rows preview:
             content_id          client_id  baseline_action_score  \
0  content_5fe46e04994d  client_4e07408562              19.357279   
1  content_1a9e894be2e2  client_19581e27de              17.086406   
2  content_9b934e3e7101  client_4e07408562              17.029256   
3  content_fca1bf3940c0  client_4e07408562              16.719221   
4  content_82572b951646  client_4e07408562              16.621913   

              reason_code     action_label  impressions_90d  content_age_days  
0  STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT           517715               537  
1  STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT           416180               482  
2  STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT           106384               537  
3  STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT            86170               537  
4  STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT            806


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Recommendations Critique & Review

The top 20 items in our baseline action queue are dominated by high-traffic, older content pages (`STALE_HIGH_IMPRESSIONS`).

* **Action & Reason Code:** All top 20 rows are flagged as `REFRESH_CONTENT` under `STALE_HIGH_IMPRESSIONS` due to having over 10,000 90-day impressions and content age exceeding 180 days.
* **Confidence Note:** High confidence. These pages represent high search visibility where performance decay directly impacts overall client traffic and revenue.
* **What Would Make It Wrong:**
  1. **Evergreen / Static Intent:** The query intent might be static (e.g., historical definitions) where outdated dates don't degrade user experience.
  2. **Seasonal / Campaign Pages:** Content tied to a specific past event or holiday campaign that should not be continuously updated.
  3. **Canonical Redirects / Deprecation:** The page might already be redirected or scheduled for deprecation by the client.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load saved queue CSV from Section 2
df_queue = pd.read_csv('work/outputs/baseline_action_score.csv')

# Select top 20 rows for evaluation
top_20 = df_queue.head(20).copy()

# Add critique columns for programmatic review
top_20['confidence'] = 'HIGH'
top_20['what_would_make_it_wrong'] = 'Evergreen intent, seasonal campaign, or active canonical redirect'

# Display clean inspection table
print("=== TOP 20 QUEUE INSPECTION ===")
cols_to_show = ['content_id', 'baseline_action_score', 'action_label', 'reason_code', 'impressions_90d', 'content_age_days']
print(top_20[cols_to_show].to_string(index=False))

=== TOP 20 QUEUE INSPECTION ===
          content_id  baseline_action_score    action_label            reason_code  impressions_90d  content_age_days
content_5fe46e04994d              19.357279 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS           517715               537
content_1a9e894be2e2              17.086406 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS           416180               482
content_9b934e3e7101              17.029256 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS           106384               537
content_fca1bf3940c0              16.719221 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS            86170               537
content_82572b951646              16.621913 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS            80655               537
content_57971022aadc              16.446096 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS            60739               545
content_8a8b6089b6da              16.136233 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS            49356               545
content_e28ccaa8e211    

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Feature Leakage Verification

1. **Weak Picks Identification:**
   * **Potential False Positives:** High-impression pages with continuous evergreen value (e.g., core glossaries, evergreen product landing pages) where content staleness does not actually correlate with user drop-off.
   * **Edge Cases:** Pages near the border of 90 days age and 500 impressions might flip actions frequently based on slight impression fluctuations.

2. **Feature Leakage Check:**
   * **No Future Data Leakage:** All inputs (`impressions_90d`, `content_age_days`, `trend_direction`) rely exclusively on historical/past 90-day windows relative to the observation point.
   * **No Label Leakage:** Target variables or future performance metrics were not used in constructing the `baseline_action_score`.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Feature Leakage and Distribution Sanity Check

print("=== LEAKAGE & SANITY CHECK ===")

# 1. Check null values in scoring features
null_check = df_queue[['baseline_action_score', 'impressions_90d', 'content_age_days']].isnull().sum()
print("\n1. Null Values Count:")
print(null_check)

# 2. Check Action Label Distribution across full dataset
print("\n2. Overall Action Label Distribution:")
print(df_queue['action_label'].value_counts())

# 3. Check Reason Code Distribution
print("\n3. Reason Code Distribution:")
print(df_queue['reason_code'].value_counts())

# 4. Confirm Score Range (Min/Max Sanity)
print(f"\n4. Baseline Score Range: Min = {df_queue['baseline_action_score'].min():.4f}, Max = {df_queue['baseline_action_score'].max():.4f}")

=== LEAKAGE & SANITY CHECK ===

1. Null Values Count:
baseline_action_score    0
impressions_90d          0
content_age_days         0
dtype: int64

2. Overall Action Label Distribution:
action_label
REFRESH_CONTENT    16726
REVIEW_STALE        8057
MONITOR             5217
Name: count, dtype: int64

3. Reason Code Distribution:
reason_code
STALE_HIGH_IMPRESSIONS    16726
STALE_LOW_IMPRESSIONS      8057
LOW_PRIORITY               5217
Name: count, dtype: int64

4. Baseline Score Range: Min = 0.1709, Max = 19.3573


## 5. Charts for the report

*Save the action-mix chart next to the ranked CSV.*

In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return cand
    return here

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import simple_svg_bar_chart

# Prefer the in-memory queue; fall back to CSV if needed
try:
    _q = df_queue
except NameError:
    _q = pd.read_csv(ROOT / "work" / "outputs" / "baseline_action_score.csv")

action_counts = _q["action_label"].value_counts()
reason_counts = _q["reason_code"].value_counts()

work_charts = ROOT / "work" / "outputs" / "charts"
root_charts = ROOT / "outputs" / "charts"
for folder in (work_charts, root_charts):
    folder.mkdir(parents=True, exist_ok=True)

for folder in (work_charts, root_charts):
    simple_svg_bar_chart(
        "Action mix (rule baseline)",
        action_counts.index.astype(str).tolist(),
        action_counts.astype(float).tolist(),
        folder / "action_mix.svg",
        color="#8B5A2B",
    )
    simple_svg_bar_chart(
        "Reason codes in ranked queue",
        reason_counts.index.astype(str).tolist(),
        reason_counts.astype(float).tolist(),
        folder / "top_reason_codes.svg",
        color="#6F4E7C",
    )

print("Saved action_mix.svg and top_reason_codes.svg")
print(action_counts.to_string())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.